# Analyse exploratoire : la dépression chez les étudiants

Dans ce notebook, j'explore le **Student Depression Dataset** (Kaggle), qui décrit 4 464 étudiants et indique s'ils déclarent être en dépression.

L'objectif n'est pas encore de prédire quoi que ce soit, mais de comprendre les données : leur structure, leur qualité, et comment le taux de dépression varie selon le profil des étudiants.

Outils utilisés : pandas et numpy.

In [1]:
import numpy as np
import pandas as pd

## Partie 1 — Découvrir les données

In [2]:
data = pd.read_csv("../data/Student_Depression_Dataset.csv")
data.head()  # j'affiche les 5 premières lignes pour voir à quoi ressemble le DataFrame

,Gender,Age,Family_background,Relationship_status,Marital_status,Living_style,Current_semester,CGPA,friend_circle,Health_condition,Cost_without_study,Home_division,Depression
0,Female,22,Middle class,Single,Unmarried,Bachelor,8,3.43,Yes,Critical,10113,Chittagong,Yes
1,Male,19,Middle class,Single,Unmarried,Bachelor,2,3.70,Yes,Abnormal,13815,Chittagong,Yes
2,Male,23,Middle class,Single,Unmarried,Family,8,3.28,No,Abnormal,15450,Dhaka,Yes
3,Male,20,Middle class,Complicated,Unmarried,Family,2,3.77,No,Good,17413,Dhaka,No
4,Female,21,Middle class,Single,Unmarried,Family,1,4.00,No,Abnormal,16693,Dhaka,Yes


In [3]:
data.shape

(4464, 13)

Le dataset contient 4 464 lignes (une par étudiant) et 13 colonnes.

In [4]:
data.dtypes

Gender                     str
Age                      int64
Family_background          str
Relationship_status        str
Marital_status             str
Living_style               str
Current_semester         int64
CGPA                   float64
friend_circle              str
Health_condition           str
Cost_without_study       int64
Home_division              str
Depression                 str
dtype: object

Les colonnes en int64 et float64 sont numériques (`Age`, `Current_semester`, `CGPA`, `Cost_without_study`), les autres contiennent du texte.

In [5]:
print("Lignes en double :", data.duplicated().sum())
print("Valeurs manquantes :", data.isna().sum().sum())

Lignes en double : 0
Valeurs manquantes : 0


Il n'y a aucune ligne en double et aucune valeur manquante.

In [6]:
colonnes_texte = ["Gender", "Family_background", "Relationship_status", "Marital_status",
                  "Living_style", "friend_circle", "Health_condition", "Home_division", "Depression"]

for colonne in colonnes_texte:
    print(data[colonne].value_counts(), "\n")

Gender
Female    2462
Male      2002
Name: count, dtype: int64 

Family_background
Middle class          2642
Lower Middle class    1151
Rich class             671
Name: count, dtype: int64 

Relationship_status
Single               2803
In a Relationship    1457
Complicated           204
Name: count, dtype: int64 

Marital_status
Unmarried    3619
Married       725
Separated     120
Name: count, dtype: int64 

Living_style
Bachelor    2449
Family      2015
Name: count, dtype: int64 

friend_circle
Yes    3215
No     1249
Name: count, dtype: int64 

Health_condition
Normal      1787
Good        1162
Abnormal     881
Critical     634
Name: count, dtype: int64 

Home_division
Dhaka         1527
Khulna         548
Rajshahi       488
Sylhet         445
Chittagong     400
Rangpur        363
Barisal        355
Mymensingh     338
Name: count, dtype: int64 

Depression
No     3263
Yes    1201
Name: count, dtype: int64 



Il y a un peu plus de femmes (2 462) que d'hommes (2 002), la majorité des étudiants sont célibataires et non mariés, et Dhaka est de loin la région la plus représentée. 1 201 étudiants ont répondu Yes à `Depression`.

## Partie 2 — Statistiques de base

In [7]:
data.describe()

,Age,Current_semester,CGPA,Cost_without_study
count,4464.000000,4464.000000,4464.000000,4464.000000
mean,22.619176,5.401210,3.412753,11242.978271
std,2.022980,2.578476,0.385605,6640.144019
min,17.000000,0.000000,2.250000,0.000000
25%,21.000000,3.000000,3.150000,6100.000000
50%,23.000000,5.000000,3.490000,9500.000000
75%,24.000000,7.000000,3.700000,15439.000000
max,28.000000,13.000000,4.000000,32000.000000


In [8]:
colonnes_num = ["Age", "Current_semester", "CGPA", "Cost_without_study"]
data[colonnes_num].median()

Age                     23.00
Current_semester         5.00
CGPA                     3.49
Cost_without_study    9500.00
dtype: float64

`describe()` ne donne les statistiques que pour les colonnes numériques :

- l'âge va de 17 à 28 ans, avec une moyenne de 22,6 ans et une médiane de 23 ans ;
- le semestre va de 0 à 13 (médiane : 5) ;
- la CGPA moyenne est de 3,41, avec un minimum de 2,25 et un maximum de 4 (médiane : 3,49) ;
- le coût hors études varie énormément, de 0 à 32 000 (médiane : 9 500). Je ne connais pas l'unité de cette colonne.

### La CGPA avec numpy

In [9]:
tableau = data["CGPA"].to_numpy()

print("Moyenne avec numpy :", tableau.mean())
print("Moyenne avec pandas :", data["CGPA"].mean())
print("Écart-type avec numpy :", tableau.std())
print("Écart-type avec pandas :", data["CGPA"].std())

Moyenne avec numpy : 3.4127531362007164
Moyenne avec pandas : 3.4127531362007164
Écart-type avec numpy : 0.38556144881787263
Écart-type avec pandas : 0.38560464172102926


Les moyennes sont identiques. Les écarts-types sont très légèrement différents parce que numpy divise par n, alors que pandas divise par n − 1 (écart-type d'un échantillon). Avec 4 464 lignes, la différence est minuscule.

In [10]:
data["Depression"].value_counts(normalize=True)

Depression
No     0.730959
Yes    0.269041
Name: proportion, dtype: float64

26,9 % des étudiants ont répondu Yes à `Depression`.

In [11]:
data["Depression_bin"] = data["Depression"].map({"Yes": 1, "No": 0})
data.head()

,Gender,Age,Family_background,Relationship_status,Marital_status,Living_style,Current_semester,CGPA,friend_circle,Health_condition,Cost_without_study,Home_division,Depression,Depression_bin
0,Female,22,Middle class,Single,Unmarried,Bachelor,8,3.43,Yes,Critical,10113,Chittagong,Yes,1
1,Male,19,Middle class,Single,Unmarried,Bachelor,2,3.70,Yes,Abnormal,13815,Chittagong,Yes,1
2,Male,23,Middle class,Single,Unmarried,Family,8,3.28,No,Abnormal,15450,Dhaka,Yes,1
3,Male,20,Middle class,Complicated,Unmarried,Family,2,3.77,No,Good,17413,Dhaka,No,0
4,Female,21,Middle class,Single,Unmarried,Family,1,4.00,No,Abnormal,16693,Dhaka,Yes,1


J'ajoute une colonne `Depression_bin` qui vaut 1 pour Yes et 0 pour No. Comme elle ne contient que des 0 et des 1, sa moyenne donne directement la proportion de Yes.

In [12]:
print(round(data["Depression_bin"].mean() * 100, 1), "%")

26.9 %


On retrouve bien 26,9 %. Je réutilise cette colonne dans toute la partie 4.

## Partie 3 — Qualité des données

In [13]:
data[colonnes_num].agg(["min", "max"])

,Age,Current_semester,CGPA,Cost_without_study
min,17,0,2.25,0
max,28,13,4.00,32000


L'âge et la CGPA ont des valeurs logiques. Par contre, deux valeurs me paraissent suspectes : un semestre égal à 0 et un coût hors études égal à 0.

In [14]:
print("Étudiants au semestre 0 :", (data["Current_semester"] == 0).sum())
print("Étudiants avec un coût de 0 :", (data["Cost_without_study"] == 0).sum())

Étudiants au semestre 0 : 42
Étudiants avec un coût de 0 : 2


42 étudiants sont au semestre 0 et 2 ont un coût hors études de 0, soit environ 1 % des lignes. Je les garde : le semestre 0 correspond peut-être à des étudiants qui viennent d'arriver, et un coût nul à des étudiants entièrement pris en charge par leur famille. Comme ça représente peu de lignes, ça ne change pas les résultats de cette analyse, mais il faudrait vérifier ce que signifient ces valeurs avant d'entraîner un modèle.

## Partie 4 — Sélections et filtres

### Les étudiants de Dhaka

In [15]:
data["Home_division"].value_counts()

Home_division
Dhaka         1527
Khulna         548
Rajshahi       488
Sylhet         445
Chittagong     400
Rangpur        363
Barisal        355
Mymensingh     338
Name: count, dtype: int64

In [16]:
data["Home_division"].value_counts(normalize=True)

Home_division
Dhaka         0.342070
Khulna        0.122760
Rajshahi      0.109319
Sylhet        0.099686
Chittagong    0.089606
Rangpur       0.081317
Barisal       0.079525
Mymensingh    0.075717
Name: proportion, dtype: float64

1 527 étudiants viennent de Dhaka, soit 34,2 % du total.

### Les 10 étudiants qui ont le coût hors études le plus élevé

In [17]:
data.sort_values("Cost_without_study", ascending=False)[["Gender", "Age", "CGPA", "Depression"]].head(10)

,Gender,Age,CGPA,Depression
2212,Male,21,3.75,No
4075,Female,21,3.75,No
4376,Male,21,3.84,No
4403,Female,26,3.74,No
4326,Male,23,3.73,Yes
4401,Female,26,3.65,No
4386,Male,25,3.85,No
3315,Female,18,3.24,No
4079,Female,21,4.00,No
4256,Male,21,4.00,No


Parmi les 10 étudiants qui ont le coût le plus élevé, un seul est en dépression, et la plupart ont une bonne CGPA (souvent au-dessus de 3,7). Sur seulement 10 personnes, je ne peux pas en tirer de conclusion.

### CGPA des étudiants déprimés et non déprimés

In [18]:
deprimes = data[data["Depression_bin"] == 1]
non_deprimes = data[data["Depression_bin"] == 0]

print("CGPA moyenne des déprimés :", deprimes["CGPA"].mean().round(2))  # round arrondit à 2 chiffres après la virgule
print("CGPA moyenne des non déprimés :", non_deprimes["CGPA"].mean().round(2))

CGPA moyenne des déprimés : 3.35
CGPA moyenne des non déprimés : 3.44


La CGPA moyenne est de 3,35 pour les étudiants déprimés et de 3,44 pour les autres. L'écart existe mais il est faible (moins de 0,1 point).

### Le cercle d'amis

In [19]:
avec_amis = data[data["friend_circle"] == "Yes"]
sans_amis = data[data["friend_circle"] == "No"]

print("Taux de dépression avec un cercle d'amis :", round(avec_amis["Depression_bin"].mean() * 100, 1), "%")
print("Taux de dépression sans cercle d'amis :", round(sans_amis["Depression_bin"].mean() * 100, 1), "%")

Taux de dépression avec un cercle d'amis : 24.0 %
Taux de dépression sans cercle d'amis : 34.3 %


Parmi ceux qui ont un cercle d'amis, 24 % sont en dépression, contre 34,3 % pour ceux qui n'en ont pas. C'est une différence de 10 points.

### L'état de santé

In [20]:
for etat in ["Good", "Normal", "Abnormal", "Critical"]:
    groupe = data[data["Health_condition"] == etat]
    print(etat, ":", round(groupe["Depression_bin"].mean() * 100, 1), "%")

Good : 0.0 %
Normal : 4.6 %
Abnormal : 54.9 %
Critical : 100.0 %


Le taux de dépression passe de 0 % pour les étudiants en bonne santé à 100 % pour ceux en état critique. Ce résultat est bizarre : dans la réalité, ce ne serait jamais aussi tranché. Soit `Health_condition` inclut déjà la santé mentale, soit les données sont en partie générées. Je garde ce point en tête pour la suite.

### Les étudiants avec une CGPA ≥ 3,8

In [21]:
haut_cgpa = data[data["CGPA"] >= 3.8]

print("Nombre d'étudiants :", len(haut_cgpa))
print("Taux de dépression :", round(haut_cgpa["Depression_bin"].mean() * 100, 1), "%")

Nombre d'étudiants : 735
Taux de dépression : 21.8 %


735 étudiants ont une CGPA supérieure ou égale à 3,8. Parmi eux, 21,8 % sont en dépression, alors que le taux global est de 26,9 %.

### Les célibataires qui vivent seuls

In [22]:
celib_seuls = data[(data["Relationship_status"] == "Single") & (data["Living_style"] == "Bachelor")]
femmes = celib_seuls[celib_seuls["Gender"] == "Female"]
hommes = celib_seuls[celib_seuls["Gender"] == "Male"]

print("Femmes célibataires qui vivent seules :", round(femmes["Depression_bin"].mean() * 100, 1), "%")
print("Hommes célibataires qui vivent seuls :", round(hommes["Depression_bin"].mean() * 100, 1), "%")

Femmes célibataires qui vivent seules : 38.6 %
Hommes célibataires qui vivent seuls : 32.7 %


Chez les célibataires qui vivent seuls, le taux de dépression est de 38,6 % pour les femmes et de 32,7 % pour les hommes. Dans les deux cas, c'est au-dessus du taux global (26,9 %).

### Le milieu familial

In [23]:
lower = data[data["Family_background"] == "Lower Middle class"]
rich = data[data["Family_background"] == "Rich class"]

print("Lower Middle class :", round(lower["Depression_bin"].mean() * 100, 1), "%")
print("Rich class :", round(rich["Depression_bin"].mean() * 100, 1), "%")

Lower Middle class : 30.6 %
Rich class : 17.6 %


Les étudiants issus d'un milieu « Lower Middle class » sont 30,6 % à être en dépression, contre 17,6 % pour ceux de « Rich class ».

## Partie 5 — Conclusion

### Observations

- 26,9 % des étudiants du dataset déclarent être en dépression.
- L'état de santé est la variable la plus liée à la dépression : de 0 % (Good) à 100 % (Critical).
- Ne pas avoir de cercle d'amis va avec un taux plus élevé : 34,3 % contre 24 %.
- Le milieu familial compte aussi : 30,6 % en « Lower Middle class » contre 17,6 % en « Rich class ».
- La CGPA change peu entre déprimés et non déprimés (3,35 contre 3,44).

### Limites

- Ce sont des corrélations, pas des causes. Par exemple, on ne sait pas si ne pas avoir d'amis rend dépressif, ou si la dépression pousse à s'isoler.
- On ne sait pas comment la dépression a été mesurée (probablement un questionnaire rempli par les étudiants), ce n'est donc pas forcément un diagnostic.
- Les étudiants viennent tous du Bangladesh : les résultats ne sont pas directement transposables à d'autres pays, comme la France.
- Je ne connais pas l'unité de `Cost_without_study`, et le lien parfait entre `Health_condition` et la dépression me fait douter de la fiabilité de certaines colonnes.

### Pour un futur modèle

Les colonnes qui me semblent les plus utiles sont `friend_circle`, `Family_background`, `Relationship_status`, `Living_style` et `Gender`. `Health_condition` prédirait presque parfaitement la dépression, mais justement, c'est suspect : le modèle risquerait de « tricher » avec une information qui contient peut-être déjà la réponse. Il faudrait comprendre ce que mesure vraiment cette colonne avant de l'utiliser. La CGPA, elle, semble peu utile.

## Bonus

### Calcul de l'écart-type de la CGPA à la main

On commence par récupérer les valeurs de la CGPA dans un tableau numpy.

In [24]:
cgpa = data["CGPA"].to_numpy()
moyenne = np.mean(cgpa)
print(moyenne)

3.4127531362007164


In [25]:
def ecart_type(valeurs: np.ndarray, moyenne: float) -> float:
    somme = 0.0  # on initialise la somme avant la boucle

    for i in range(len(valeurs)):
        somme += (valeurs[i] - moyenne) ** 2  # on ajoute chaque écart au carré

    variance = somme / len(valeurs)  # la variance est la moyenne des écarts au carré
    return np.sqrt(variance)  # l'écart-type est la racine de la variance

In [26]:
print("Mon écart-type :", ecart_type(cgpa, moyenne))
print("np.std :", np.std(cgpa))

Mon écart-type : 0.3855614488178719
np.std : 0.38556144881787263


On obtient la même valeur qu'avec `np.std`, donc ma fonction marche (elle divise par n, comme numpy).

### Fonction taux_depression

In [27]:
def taux_depression(df: pd.DataFrame) -> float:
    return df["Depression_bin"].mean() * 100

In [28]:
print("Taux global :", round(taux_depression(data), 1), "%")
print("Sans cercle d'amis :", round(taux_depression(sans_amis), 1), "%")
print("CGPA ≥ 3,8 :", round(taux_depression(haut_cgpa), 1), "%")

Taux global : 26.9 %
Sans cercle d'amis : 34.3 %
CGPA ≥ 3,8 : 21.8 %


Avec cette fonction, je n'ai plus besoin de réécrire le même calcul à chaque fois.